In [ ]:
import subprocess, sys, os, time, glob, shutil, hashlib, json
T0 = time.time()
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode
sh("apt-get -qq update && apt-get -qq install -y openjdk-8-jre-headless")
sh("update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java")
sh(f"{sys.executable} -m pip install -q 'transformers==5.20.2' 'peft==0.21.2' 'trl==0.29.1' accelerate pycocoevalcap sacrebleu bert_score 2>&1 | tail -3")
sh(f"{sys.executable} -m pip uninstall -y -q torchao")
r = subprocess.run([sys.executable, "-c", "import trl, transformers, peft; print(trl.__version__, transformers.__version__, peft.__version__)"], capture_output=True, text=True)
print(r.stdout, r.stderr[-500:], flush=True)
assert r.stdout.split() == ["0.29.1", "5.20.2", "0.21.2"], "DUNG: lech phien ban"
W = "/kaggle/working"; S = f"{W}/s"; OUT = f"{W}/out"
os.makedirs(S, exist_ok=True); os.makedirs(OUT, exist_ok=True)
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
MD5 = {'grpo_luoi.py': '92d0a9bc6b8cae8a480aef08f99fb9d8', 'grpo_spice.py': '8489108147c8d1ed80c3dc67c8ad3d3f', 'build_branch_data.py': '619e63e123a6dbf60086e65ee94a3912', 'grpo_codien.py': '5499d6041bf0fb7c780099abcaa27f32', 'grpo_ppo.py': 'e3330d4469f6b60c9e3795e2d46b83db'}
for f, h in MD5.items():
    k = [p for p in glob.glob(f"/kaggle/input/**/{f}", recursive=True) if "/notebooks/" not in p]
    assert len(k) == 1, f"DUNG: can dung mot {f}, thay {k}"
    shutil.copyfile(k[0], f"{S}/{f}"); assert md5(f"{S}/{f}") == h, f"DUNG: {f} lech ban may nha"
open(f"{S}/thu.py", "w").write('"""python thu.py <file script> <cờ...>: chạy script như __main__ nhưng nạp mô hình tí hon."""\nimport os, sys, runpy\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport torch\nimport grpo_spice as GS\nfrom transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration\nTINY = "trl-internal-testing/tiny-Qwen2_5_VLForConditionalGeneration"\nclass AnyEq(int):\n    def __eq__(self, o): return True\n    __hash__ = int.__hash__\ndef nap(a):\n    return AutoProcessor.from_pretrained(TINY), Qwen2_5_VLForConditionalGeneration.from_pretrained(TINY, dtype=torch.float32), torch.float32\nGS.nap = nap\nGS._n_s1 = lambda p: AnyEq(0)\nf = sys.argv[1]\nsys.argv = sys.argv[1:]\nrunpy.run_path(f, run_name="__main__")\n')
import importlib.util
SPL = list(importlib.util.find_spec("pycocoevalcap").submodule_search_locations)[0] + "/spice/lib"
for j, sz in {"stanford-corenlp-3.6.0.jar": 7270741, "stanford-corenlp-3.6.0-models.jar": 378116526}.items():
    src = glob.glob(f"/kaggle/input/**/{j}", recursive=True); assert src, f"DUNG: thieu {j}"
    shutil.copyfile(src[0], f"{SPL}/{j}"); assert os.path.getsize(f"{SPL}/{j}") == sz
# bundle nhỏ: dữ liệu thật (câu nhắc, câu chuẩn, OCR), ảnh xám 112x224, adapter giả r=8
REAL = next(r for r, d, f in os.walk("/kaggle/input") if "adapter_s1_seed101" in d and "images" in d)
B = f"{W}/bundle_thu"
os.makedirs(f"{B}/adapter_s1_seed101", exist_ok=True)
for f in ("p1_train_rows.jsonl", "p1_val_rows.jsonl", "ocr.jsonl"):
    shutil.copyfile(f"{REAL}/{f}", f"{B}/{f}")
from PIL import Image
for l in open(f"{B}/p1_train_rows.jsonl"):
    p = f"{B}/" + json.loads(l)["image"]; os.makedirs(os.path.dirname(p), exist_ok=True)
    if not os.path.exists(p): Image.new("RGB", (112, 224), (200, 200, 200)).save(p)
json.dump({"r": 8, "lora_alpha": 16, "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"]}, open(f"{B}/adapter_s1_seed101/adapter_config.json", "w"))
print("chuan bi xong", round(time.time() - T0), "s", flush=True)


In [ ]:
RUNS = ['scst:cider:101', 'vanilla:spice:101', 'a2c:spice:101', 'grpo:spice:101:loss=dr_grpo,scale=none,tag=drg', 'grpo:spice:101:beta=0,tag=dapo0', 'grpo:bleu4:101', 'grpo:chrf:101', 'grpo:spider:101', 'grpo:bertscore:101', 'rft:spice:101', 'cem:spice:101', 'grpo:spice:101:scale=batch,tag=sb', 'awr:spice:101', 'raml:spice:101', 'rank:spice:101', 'mad:spice:101', 'grpo:spice:101:mu=2,steps=1000,tag=mu2', 'grpo:spice:101:mu=2,eps=0.28,steps=1000,tag=ch', 'grpo:spice:101:mu=2,is=sequence,steps=1000,tag=gspo', 'grpo:spice:101:G=4,tag=g4', 'grpo:spice:101:G=16,tag=g16']
CD = ("vanilla", "mad", "rft", "cem", "awr", "raml", "rank")
CO = {"beta": "--beta", "loss": "--loss-type", "scale": "--scale-rewards", "mu": "--mu", "lam": "--lam", "is": "--is-level",
      "eps": "--eps-high", "steps": "--max-steps", "G": "--G"}
def opt_of(r):
    x = r.split(":")[3:]
    return dict(kv.split("=") for kv in x[0].split(",")) if x else {}
def tep_cua(r):   # chép đúng luật chọn script của make_nb.py
    al, rw = r.split(":")[:2]; ok = opt_of(r)
    if al in ("ppo", "a2c"): return "grpo_ppo.py"
    return "grpo_codien.py" if al in CD or set(ok) - {"tag"} or (al != "grpo" and rw != "spice") else "grpo_luoi.py"
for f, a in (("grpo_luoi.py", "--selftest-algo"), ("grpo_codien.py", "--selftest-algo"), ("grpo_ppo.py", "--selftest-algo")):
    r = subprocess.run([sys.executable, f, a], cwd=S, capture_output=True, text=True)
    print(f, a, "ma", r.returncode, (r.stdout + r.stderr)[-300:], flush=True); assert r.returncode == 0
KQ = {}
for r in RUNS:
    al, rw, sd = r.split(":")[:3]; ok = opt_of(r)
    tag = f"{al}_{rw}_{sd}" + ("_" + ok["tag"] if "tag" in ok else "")
    o = f"{OUT}/{tag}"; shutil.rmtree(o, ignore_errors=True)
    cmd = [sys.executable, "thu.py", tep_cua(r), "--train", "--no-q4", "--bs", "4", "--accum", "4", "--algo", al, "--reward", rw,
           "--seed", sd, "--bundle", B, "--merged", "x", "--out", o]
    for k, v in ok.items():
        if k not in ("tag", "steps"): cmd += [CO[k], v]
    cmd += ["--max-steps", "2"]
    t = time.time()
    with open(f"{OUT}/{tag}.log", "w") as lg:
        p = subprocess.run(cmd, cwd=S, stdout=lg, stderr=subprocess.STDOUT, text=True,
                           env={**os.environ, "CUDA_VISIBLE_DEVICES": "", "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1"})
    L = open(f"{OUT}/{tag}.log", errors="ignore").read().splitlines()
    hp = f"{o}/log_history.json"
    h = json.load(open(hp)) if os.path.exists(hp) else []
    buoc = max([x.get("step", 0) for x in h] or [0])
    dat = p.returncode == 0 and buoc >= 2 and not any("Traceback" in l for l in L)
    KQ[r] = dict(ma=p.returncode, buoc=buoc, giay=round(time.time() - t), dat=dat)
    print(f"{(time.time()-T0)/60:5.1f} ph · {'DAT ' if dat else 'LOI '} {r} · ma {p.returncode} · buoc {buoc} · {KQ[r]['giay']} s", flush=True)
    for l in L:
        if l.startswith(("[đè cấu hình]", "[cấu hình]", "[PPO]", "[bertscore]", "[cider]")) or "num_iterations" in l: print("     ", l[:220], flush=True)
    if h:
        print("      log cuoi:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in h[-1].items()
              if k.startswith(("reward", "luoi", "loss", "kl", "clip", "step", "frac"))}, flush=True)
    if not dat: print("\n".join("      | " + l for l in L[-25:]), flush=True)
json.dump(KQ, open(f"{OUT}/ket_qua.json", "w"), indent=1, ensure_ascii=False)
shutil.make_archive(f"{W}/thu_cpu", "zip", OUT)
print("\nTONG:", sum(v["dat"] for v in KQ.values()), "/", len(KQ), "dat", flush=True)
assert all(v["dat"] for v in KQ.values()), [k for k, v in KQ.items() if not v["dat"]]
